# SMS backup cleanup — merge everything into one file
Run this signed in to the Google account that **owns** the backups. Colab runs in Google's cloud and reads your Drive directly; nothing is downloaded to your computer.

**What happens:**
1. **Merge** — every `sms-*.xml` / `calls-*.xml` backup is read and every message is written **once** into a single file in the `…-merged` folder (one for texts/MMS, one for call logs if you have any). It's a normal backup file the app can restore.
2. **Verify** — an independent second pass re-reads every original backup and proves every text, MMS (including photos) and call is in the merged file. Only if it says **VERIFIED** are the originals moved into a `…-to-delete` folder.
3. **You** delete the `…-to-delete` folder and **empty the Drive trash**.

Both passes read all your backups, so each takes roughly 1 hour per ~125 GB. If Colab disconnects, just run the cell again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Get the tool. If the repo is private, use a fine-grained read-only token:
#   !git clone https://<TOKEN>@github.com/LegitUnicorn/SMS-Cleanup.git
!rm -rf /content/SMS-Cleanup && git clone -q https://github.com/LegitUnicorn/SMS-Cleanup.git /content/SMS-Cleanup
%cd /content/SMS-Cleanup

In [ ]:
# ---- EDIT THIS: the Drive folder that holds sms-YYYYMMDDHHMMSS.xml / calls-*.xml ----
import os, re, shlex, collections

# Either a path under /content/drive/MyDrive, or just paste the folder's link from drive.google.com
BACKUP_DIR = '/content/drive/MyDrive/SMSBackupRestore'

if BACKUP_DIR.startswith('http'):
    from smscleanup.colab import resolve_folder_link
    BACKUP_DIR = resolve_folder_link(BACKUP_DIR)  # also checks you own the folder
BACKUP_DIR = BACKUP_DIR.rstrip('/')
MERGED_DIR    = BACKUP_DIR + '-merged'      # the ONE merged file ends up here (keep this folder)
TO_DELETE_DIR = BACKUP_DIR + '-to-delete'   # verified originals are moved here (delete this folder)
OLD_WORK_DIR  = BACKUP_DIR + '-cleanup'     # only exists if you used the older analyze/apply steps
INPUTS = [BACKUP_DIR] + ([OLD_WORK_DIR] if os.path.isdir(OLD_WORK_DIR) else [])
INPUT_ARGS = ' '.join(shlex.quote(p) for p in INPUTS)

PAT = re.compile(r'^(sms|calls)-\d{14}.*\.xml$', re.I)

def find_backup_folders(root='/content/drive/MyDrive'):
    hits = collections.Counter()
    for d, subdirs, files in os.walk(root):
        subdirs[:] = [s for s in subdirs if not s.endswith(('-cleanup', '-merged', '-to-delete'))]
        n = sum(1 for f in files if PAT.match(f))
        if n:
            hits[d] = n
    return hits

files = sorted(f for f in os.listdir(BACKUP_DIR) if PAT.match(f)) if os.path.isdir(BACKUP_DIR) else []
if not files:
    print(f'No sms-/calls- backup files in {BACKUP_DIR}')
    print('Searching your whole Drive for them (can take a minute)...')
    hits = find_backup_folders()
    for d, n in hits.most_common():
        print(f'  {n:5d} backup files in {d}')
    if hits:
        print('\n>>> Set BACKUP_DIR above to the folder with the most files and re-run this cell.')
    else:
        print('Nothing found under My Drive. If the backups are in "Shared with me", '
              'add a shortcut to that folder in My Drive first.')
else:
    total = sum(os.path.getsize(os.path.join(BACKUP_DIR, f)) for f in files)
    print(f'{len(files)} backup files, {total/1e9:.1f} GB in {BACKUP_DIR}')
    for f in files[:5] + (['...'] if len(files) > 10 else []) + files[-5:]:
        print('  ', f)
    print('Inputs:', INPUTS)
    print('Merged file will be written to:', MERGED_DIR)

## 1. Merge
Reads the backups (newest first) and writes one deduplicated file. Your original backups are **not changed**. Where the same message is in many backups, the copy from the newest backup is kept (latest read status / contact name).

At the end it re-reads the new file and checks every record came out identical (`self-check passed`).

In [ ]:
!python -m smscleanup merge $INPUT_ARGS --out-dir "$MERGED_DIR" --tmp-dir /content/tmp

## 2. Verify (and gather the originals for deletion)
Ignores everything the merge step did and re-reads **every original backup** from Drive, checking each message against the merged file:
* **exact** — an identical copy is in the merged file
* **details differ** — the same message is there; only things like the read flag or contact name differ (listed by field)
* **MISSING** — not in the merged file: saved to `verify-missing-*.xml` in the merged folder, and nothing is moved

Only if the result is **VERIFIED** are the original backups moved into the `…-to-delete` folder.

In [ ]:
!python -m smscleanup verify-merged $INPUT_ARGS --out-dir "$MERGED_DIR" --move-sources-to "$TO_DELETE_DIR"

## 3. Free the space (only after VERIFIED)
1. In drive.google.com delete the **`…-to-delete`** folder (and the old **`…-cleanup`** folder, if you have one).
2. Open **Trash → Empty trash**. Storage is only released after this (the quota meter can take a few hours to update).
3. **Keep the `…-merged` folder.** Its `sms-…-merged.xml` (and `calls-…-merged.xml`) holds every message from every backup. To restore it, point the backup app's restore at that file.

Keep the merged file outside the app's own backup folder so the app's "delete old backups" setting can never remove it.